# Tap-derived Kármán prior: derivation and executable demonstration

This notebook documents the analytical prior used by the reported **V1-only radial-trust ModalPINN** experiment. It is deliberately independent of TensorFlow and neural-network training. It loads the frozen parameter archive, runs the numerical Lamb–Oseen vortex street, evaluates the closed-form first harmonic, and reconstructs the exact downstream gate used during training.

Information boundary: the parameter chain uses the same 32 cylinder-pressure histories supplied to the neural reconstruction, the known geometry, and classical vortex-street relations. No interior CFD velocity is fitted. The CFD field is used only later, outside this notebook, for evaluation.

## 1. Physical model

Two staggered vortex rows are placed at $y=\pm h/2$. Their streamwise spacing is $a$, the lower row is offset by $a/2$, and the classical stable spacing ratio is

$$\frac{h}{a}=0.281, \qquad a=\frac{2\pi U_c}{\omega_0}. $$

Each vortex is regularised as a Lamb–Oseen core. For a vortex centred at $\mathbf{x}_v$, with $\Delta x=x-x_v$, $\Delta y=y-y_v$ and $r^2=\Delta x^2+\Delta y^2$, the induced Cartesian velocity is

$$u=-\frac{\Gamma\,\Delta y}{2\pi r^2}\left(1-e^{-r^2/r_c^2}\right), \qquad v=\frac{\Gamma\,\Delta x}{2\pi r^2}\left(1-e^{-r^2/r_c^2}\right).$$

The core grows downstream according to

$$r_c^2(x_v)=r_0^2+\frac{4\nu\max(x_v-x_f,0)}{U_c}, \qquad \nu=1/Re.$$

The numerical reference street also includes Milne–Thomson image vortices so that the cylinder is a streamline, a potential-flow dipole for the mean field, and a smooth formation envelope $[1+\tanh((x-x_f)/\delta_x)]/2$.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np

def find_repository():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        prior = candidate / 'results/data/geometry/street_prior_Ntap32.npz'
        if prior.exists():
            return candidate
    raise FileNotFoundError('Run this notebook inside the ModalPINN2.0 repository.')

REPO = find_repository()
VENDOR = REPO / 'results/code/vendor'
sys.path.insert(0, str(VENDOR))
from street_prior import Street, cf_modes_uv, HA_RATIO, NU, R_C  # noqa: E402

PRIOR_FILE = REPO / 'results/data/geometry/street_prior_Ntap32.npz'
archive = np.load(PRIOR_FILE)
parameter_names = ('Gamma', 'Uc', 'xf', 'r0', 'omega', 'phase',
                   'amp_scale', 'scale_p', 'ramp', 'delta')
prm = {name: float(archive[name]) for name in parameter_names}

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10, 'axes.titlesize': 11,
    'axes.labelsize': 10, 'axes.spines.top': False,
    'axes.spines.right': False, 'savefig.bbox': 'tight'
})
print('Loaded:', PRIOR_FILE.relative_to(REPO))
for name in parameter_names:
    print(f'{name:>12s} = {prm[name]: .6f}')

## 2. How the 32 taps determine the archive

1. The tap pressures are integrated around the cylinder to obtain pressure-drag and lift histories.
2. A sinusoid is fitted to lift to estimate $\omega_0$.
3. Each tap signal is projected onto $\{1,\cos(\omega_0t),\sin(\omega_0t)\}$, giving its first pressure harmonic.
4. Circulation $\Gamma$ is obtained by solving the Kármán drag relation together with

$$U_c=1-\frac{\Gamma}{\sqrt{8}a}, \qquad a=\frac{2\pi U_c}{\omega_0}.$$

5. Formation position $x_f$, initial core radius $r_0$, and phase are chosen by matching the image-street surface-pressure harmonic to the measured tap harmonic.
6. The differentiable closed-form harmonic is amplitude- and phase-aligned to the numerical analytical street at wake points. This is street-to-street calibration; it does not use the CFD interior field.

The original circulation calculation used $C_{D,\mathrm{total}}\simeq C_{D,p}/0.75$. A later full-wall stress integration measured $\overline C_{D,p}=0.9890$, $\overline C_{D,\nu}=0.3400$, and $\overline C_D=1.3291$, so the measured pressure fraction is **74.42%**. The rounded 75% assumption differs by 0.58 percentage points.

In [ ]:
a = 2 * np.pi * prm['Uc'] / prm['omega']
h = HA_RATIO * a
period = 2 * np.pi / prm['omega']
street = Street(prm['Gamma'], prm['Uc'], x_f=prm['xf'], r0=prm['r0'],
                phase=prm['phase'], omega=prm['omega'], ramp=prm['ramp'])

print(f'Period T       = {period:.4f}')
print(f'Spacing a     = {a:.4f} D')
print(f'Row distance h= {h:.4f} D')
print(f'h/a           = {h/a:.4f}')
print(f'Pressure drag = {float(archive["CD_pressure"]):.4f}')
print(f'Tap p1 corr.  = {float(archive["tap_p1_corr"]):.4f}')
print(f'Closed/numeric correlation = {float(archive["cf_corr_vs_numeric"]):.4f}')
assert abs(h / a - 0.281) < 1e-12
assert float(archive['cf_corr_vs_numeric']) > 0.95

## 3. Run the vortex street

The next cell evaluates the same numerical `Street` class used when deriving the prior. Red and blue markers show opposite-signed staggered vortices. Streamlines and colour show the analytical velocity field at one phase.

In [ ]:
xg = np.linspace(-1.0, 8.0, 220)
yg = np.linspace(-2.3, 2.3, 120)
X, Y = np.meshgrid(xg, yg)
points = np.column_stack((X.ravel(), Y.ravel()))
u, v = street.velocity(points, t=0.0)
U, V = u.reshape(X.shape), v.reshape(X.shape)
speed = np.sqrt(U**2 + V**2)
inside = X**2 + Y**2 <= R_C**2
U = np.ma.array(U, mask=inside); V = np.ma.array(V, mask=inside)
speed = np.ma.array(speed, mask=inside)
upper, lower = street._vortex_positions(t=0.0)

fig, ax = plt.subplots(figsize=(10.5, 4.2), constrained_layout=True)
levels = np.linspace(0.0, np.nanpercentile(speed.compressed(), 98), 24)
contour = ax.contourf(X, Y, speed, levels=levels, cmap='viridis', extend='max')
ax.streamplot(xg, yg, U, V, color='white', density=1.15, linewidth=0.45, arrowsize=0.65)
show_upper = upper[(upper[:, 0] >= -1.0) & (upper[:, 0] <= 8.0)]
show_lower = lower[(lower[:, 0] >= -1.0) & (lower[:, 0] <= 8.0)]
ax.scatter(show_upper[:, 0], show_upper[:, 1], s=42, c='#C43C39', edgecolor='white',
           linewidth=0.7, label=r'upper row, $-\Gamma$', zorder=5)
ax.scatter(show_lower[:, 0], show_lower[:, 1], s=42, c='#2B6CB0', edgecolor='white',
           linewidth=0.7, label=r'lower row, $+\Gamma$', zorder=5)
ax.add_patch(Circle((0, 0), R_C, facecolor='#263238', edgecolor='white', linewidth=1.0, zorder=6))
ax.set(xlim=(-1, 8), ylim=(-2.3, 2.3), xlabel=r'$x/D$', ylabel=r'$y/D$',
       title='Numerical Lamb–Oseen street reconstructed from the frozen tap-derived parameters')
ax.set_aspect('equal')
ax.legend(loc='upper right', frameon=True, ncol=2)
cbar = fig.colorbar(contour, ax=ax, pad=0.015)
cbar.set_label(r'analytical speed $|\mathbf{u}|/U_\infty$')
plt.show()

## 4. Differentiable first harmonic used by ModalPINN

For row $j$, the closed-form contribution at harmonic $k$ is

$$B_{j,k}=s_j\frac{\Gamma}{2a}\exp\!\left[-\frac{2\pi k}{a}\operatorname{softabs}(y-y_j)\right]\exp\!\left[-\frac{(\pi k)^2r_c^2(x)}{a^2}\right]\exp\!\left[-i\frac{2\pi k(x-x_{0,j})}{a}-ik\phi\right].$$

The transverse mode is $S_{v,k}=i\,f(x)\sum_j B_{j,k}$, multiplied by the saved street-to-street amplitude calibration and converted to the one-sided ModalPINN convention. The reported experiment uses **only** $S_{v,1}$ in its trust region. The full numerical street above helps derive and visualise the prior, but it is not pasted wholesale into the network.

In [ ]:
def smootherstep01(z):
    z = np.clip(z, 0.0, 1.0)
    return z**3 * (z * (z * 6.0 - 15.0) + 10.0)

def v1_trust_gate(x, y, xstart=3.0, xwidth=0.30, ymax=2.0, ywidth=0.20):
    wx = smootherstep01((x - (xstart - xwidth)) / xwidth)
    wy_hi = 1.0 - smootherstep01((y - ymax) / ywidth)
    wy_lo = 1.0 - smootherstep01((-y - ymax) / ywidth)
    return wx * wy_hi * wy_lo

us, vs = cf_modes_uv(X.ravel(), Y.ravel(), prm, nk=1)
S_v1 = (2.0 * prm['amp_scale'] * vs[0]).reshape(X.shape)
W = v1_trust_gate(X, Y)

fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.5), constrained_layout=True, sharex=True, sharey=True)
items = [
    (np.abs(S_v1), r'Prior amplitude $|S_{v,1}|$', 'magma'),
    (W, r'V1 radial-trust gate $W(x,y)$', 'Blues'),
    (W * np.abs(S_v1), r'Prior amplitude inside the trusted region', 'magma'),
]
for ax, (field, title, cmap) in zip(axes, items):
    image = ax.pcolormesh(X, Y, field, shading='auto', cmap=cmap)
    ax.add_patch(Circle((0, 0), R_C, facecolor='#263238', edgecolor='white', linewidth=0.8))
    ax.axvline(3.0, color='white', linewidth=0.8, linestyle='--')
    ax.set(xlim=(-1, 8), ylim=(-2.3, 2.3), xlabel=r'$x/D$', title=title)
    ax.set_aspect('equal')
    fig.colorbar(image, ax=ax, pad=0.02, shrink=0.82)
axes[0].set_ylabel(r'$y/D$')
plt.show()

assert np.all(W[X >= 3.0] <= 1.0 + 1e-14)
assert np.allclose(W[(X >= 3.0) & (np.abs(Y) <= 2.0)], 1.0)

## 5. Exact reported hybrid ansatz

Let $z(x,y)$ be the free complex output of the transverse-velocity network, $f_{BC}$ the hard cylinder no-slip factor, and $W(x,y)$ the gate plotted above. The reported model evaluates

$$\widehat v_1=f_{BC}\left[(1-W)z+W\left(S_{v,1}+\rho_{\mathrm{tr}}|S_{v,1}|_\varepsilon\frac{z}{\sqrt{1+|z|^2}}\right)\right], \qquad \rho_{\mathrm{tr}}=0.60.$$

The smooth magnitude $|S|_\varepsilon=\sqrt{(\Re S)^2+(\Im S)^2+\varepsilon^2}-\varepsilon$ avoids a non-smooth absolute value inside a field differentiated by the PDE loss. In the trusted core, the correction magnitude is strictly less than $0.60|S_{v,1}|_\varepsilon$, so a non-zero prior cannot collapse exactly to zero. Outside the gate, $\widehat v_1=f_{BC}z$ is the ordinary ModalPINN output.

This distinction matters: the earlier R9 `--TrustStreet` prototype applied the street to $u$, $v$, and $p$ for every $k\ge1$. The dissertation results instead use the later `--V1RadialTrust` option described here.

## 6. What this notebook proves—and what it does not

The notebook verifies that the frozen archive is readable, the physical street runs without a neural network, the closed-form/numerical calibration recorded in the archive exceeds 0.95 correlation, and the V1-only gate matches the reported run configuration. It does **not** validate reconstruction accuracy against CFD; those comparisons are performed by the common evaluator and are collected in `results/all_results.xlsx`.

Key implementation sources:

- `results/code/vendor/street_prior.py`: tap-to-parameter chain, numerical street, and closed-form modes.
- `runs/R10_extracted/R10_v1radial_smoke/R10_smoke_radial/NN_functions.py`: exact V1 radial-trust wrapper.
- `results/code/evaluate_common.py`: shared regions and error metrics.
- Boudina CFD data: https://zenodo.org/records/5039610